# Music Annotation Project V7.5 Final Slice: Contact Sheets and Worst Crops

**Runtime:** CPU. Select **Runtime > Run all**.

This is a **reporting-only** notebook. It does not rerun reconstruction and does not change V7.5 routing or metrics. It reads the completed V7.5 run, its persistent reconstruction caches, and its evidence tables, then adds the missing visual outputs.

## Outputs added to the selected V7.5 run

- `contact_sheets/page_comparisons/`: whole-page comparisons for every cached page;
- `contact_sheets/component_review/`: ranked component sheets showing clean truth, observed input, both geometry candidates and route overlays;
- `worst_crops/preserve_false_positive/`: lowest preserve-route clean-print precision;
- `worst_crops/remove_false_positive/`: lowest remove-route clean-paper precision;
- `worst_crops/high_ambiguity/`: highest reconstruction dependency;
- `worst_crops/staff/`, `symbol/`, `mixed/`, `background/`: worst reconstruction crops using region-appropriate metrics;
- `tables/final_slice_rankings.csv` and `final_slice_manifest.json`.

Clean images are used only for post-hoc reporting and ranking. No reconstruction output is recomputed.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas matplotlib


In [ ]:
import cv2, json, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime, timezone

ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
V75_ROOT=ROOT/'reconstruction_geometry_benchmark_v7_5'
RUN_OVERRIDE=None  # Set to a complete run folder if the latest run should not be used.
MAX_WORST_CROPS=20
CROP_MARGIN=40
PAGE_DPI=145
CROP_DPI=175
SUPPORTED={'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}
METHODS=['telea','hard_raw_t80','hard_raw_t110','geometry_t80','geometry_t110']

def image_files(folder):
    folder=Path(folder)
    return sorted(p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in SUPPORTED) if folder.exists() else []
def image_map(folder): return {p.stem:p for p in image_files(folder)}
def read_csv_required(path):
    if not Path(path).exists(): raise RuntimeError(f'Missing required V7.5 table: {path}')
    return pd.read_csv(path)
def completed_run(path):
    required=[path/'benchmark_manifest.json',path/'tables'/'cache_status.csv',path/'tables'/'metrics_by_page_region.csv',path/'tables'/'surrounding_area_evidence_by_component.csv',path/'tables'/'surrounding_evidence_truth_diagnostics.csv']
    return all(p.exists() for p in required)

if RUN_OVERRIDE:
    RUN=Path(RUN_OVERRIDE)
else:
    candidates=[p for p in (V75_ROOT/'runs').iterdir() if p.is_dir() and completed_run(p)] if (V75_ROOT/'runs').exists() else []
    if not candidates: raise RuntimeError(f'No completed V7.5 run found under {V75_ROOT/"runs"}')
    RUN=max(candidates,key=lambda p:p.stat().st_mtime)

TABLES=RUN/'tables'; CONTACTS=RUN/'contact_sheets'; PAGE_CONTACTS=CONTACTS/'page_comparisons'; COMPONENT_CONTACTS=CONTACTS/'component_review'; WORST=RUN/'worst_crops'
for d in [CONTACTS,PAGE_CONTACTS,COMPONENT_CONTACTS,WORST]: d.mkdir(parents=True,exist_ok=True)
for name in ['preserve_false_positive','remove_false_positive','high_ambiguity','staff','symbol','mixed','background']:
    (WORST/name).mkdir(parents=True,exist_ok=True)

CLEAN_MAP=image_map(ROOT/'source'/'clean'); OBSERVED_MAP=image_map(ROOT/'aligned')
cache_df=read_csv_required(TABLES/'cache_status.csv')
metric_df=read_csv_required(TABLES/'metrics_by_page_region.csv')
evidence_df=read_csv_required(TABLES/'surrounding_area_evidence_by_component.csv')
truth_df=read_csv_required(TABLES/'surrounding_evidence_truth_diagnostics.csv')
component_df=evidence_df.merge(truth_df,on=['page','method','component'],how='left',validate='one_to_one')

required_cache={'page','cache'}
if not required_cache.issubset(cache_df.columns): raise RuntimeError(f'cache_status.csv lacks {sorted(required_cache-set(cache_df.columns))}')
missing_pages=sorted(set(cache_df.page)-set(CLEAN_MAP)) + sorted(set(cache_df.page)-set(OBSERVED_MAP))
if missing_pages: raise RuntimeError(f'Missing clean or observed page images: {sorted(set(missing_pages))}')
print('Selected V7.5 run:',RUN)
print('Cached pages:',cache_df.page.nunique(),'Component rows:',len(component_df))


## Cache-loading and visualisation helpers


In [ ]:
def load_page(page):
    row=cache_df[cache_df.page==page]
    if row.empty: raise KeyError(page)
    cache=Path(row.iloc[0].cache)
    if not cache.exists():
        alternatives=sorted((V75_ROOT/'persistent_cache').glob(f'{page}_*.npz'))
        if not alternatives: raise RuntimeError(f'Cache missing for {page}: {cache}')
        cache=max(alternatives,key=lambda p:p.stat().st_mtime)
    clean=cv2.imread(str(CLEAN_MAP[page]),0); observed=cv2.imread(str(OBSERVED_MAP[page]),0)
    if clean is None or observed is None: raise RuntimeError(f'Unreadable image for {page}')
    z=np.load(cache)
    missing=[x for x in ['mask']+METHODS if x not in z.files]
    if missing: raise RuntimeError(f'{page} cache lacks {missing}')
    arrays={k:z[k] for k in z.files}
    return clean,observed,arrays,cache

def route_overlay(observed,preserve,remove,ambiguous):
    rgb=cv2.cvtColor(observed,cv2.COLOR_GRAY2RGB); layer=rgb.copy()
    layer[preserve]=(25,170,25); layer[remove]=(35,105,235); layer[ambiguous]=(240,145,25)
    return cv2.addWeighted(rgb,.52,layer,.48,0)

def route_arrays(arrays,threshold):
    prefix=f'g{threshold}_'
    return {k:arrays[prefix+k]>0 for k in ['preserve','remove','ambiguous','horizontal','vertical','compact']}

def crop_bounds(row,shape,margin=CROP_MARGIN):
    H,W=shape; x=int(row.x);y=int(row.y);w=int(row.w);h=int(row.h)
    return max(0,x-margin),max(0,y-margin),min(W,x+w+margin),min(H,y+h+margin)

def show_gray(ax,image,title):
    ax.imshow(image,cmap='gray',vmin=0,vmax=255);ax.set_title(title,fontsize=9);ax.axis('off')
def save_page_sheet(page):
    clean,observed,a,cache=load_page(page)
    r80=route_arrays(a,80);r110=route_arrays(a,110)
    panels=[clean,observed,a['telea'],a['hard_raw_t80'],a['hard_raw_t110'],a['geometry_t80'],a['geometry_t110'],route_overlay(observed,r80['preserve'],r80['remove'],r80['ambiguous']),route_overlay(observed,r110['preserve'],r110['remove'],r110['ambiguous'])]
    titles=['clean truth','observed','Telea','hard raw t80','hard raw t110','geometry t80','geometry t110','routes t80','routes t110']
    fig,axes=plt.subplots(3,3,figsize=(18,24))
    for ax,image,title in zip(axes.ravel(),panels,titles):
        if image.ndim==2: show_gray(ax,image,title)
        else: ax.imshow(image);ax.set_title(title,fontsize=9);ax.axis('off')
    fig.suptitle(f'{page} | V7.5 final-slice page comparison | green preserve, blue remove, orange ambiguous',fontsize=14)
    plt.tight_layout(); path=PAGE_CONTACTS/f'{page}_v7_5_page_comparison.png';plt.savefig(path,dpi=PAGE_DPI,bbox_inches='tight');plt.close(fig);return path

def save_component_sheet(row,category,rank):
    page=str(row.page); threshold=int(str(row.method).split('t')[-1]); clean,observed,a,_=load_page(page); routes=route_arrays(a,threshold)
    x0,y0,x1,y1=crop_bounds(row,clean.shape)
    crop=lambda im:im[y0:y1,x0:x1]
    overlay=route_overlay(observed,routes['preserve'],routes['remove'],routes['ambiguous'])
    panels=[crop(clean),crop(observed),crop(a['telea']),crop(a['geometry_t80']),crop(a['geometry_t110']),crop(overlay),crop(routes['preserve'].astype(np.uint8)*255),crop(routes['remove'].astype(np.uint8)*255),crop(routes['ambiguous'].astype(np.uint8)*255)]
    titles=['clean truth','observed','Telea','geometry t80','geometry t110',f'routes {threshold}','preserve pixels','remove pixels','ambiguous pixels']
    fig,axes=plt.subplots(3,3,figsize=(15,15))
    for ax,image,title in zip(axes.ravel(),panels,titles):
        if image.ndim==2: show_gray(ax,image,title)
        else: ax.imshow(image);ax.set_title(title,fontsize=9);ax.axis('off')
    diagnostics=[]
    for field,label in [('preserved_clean_print_precision','preserve precision'),('removed_clean_paper_precision','remove precision'),('reconstruction_dependency_fraction','ambiguous fraction'),('clean_print_fraction','clean print fraction')]:
        if field in row.index and pd.notna(row[field]): diagnostics.append(f'{label}={float(row[field]):.3f}')
    fig.suptitle(f'{category} #{rank:02d} | {page} component {int(row.component)} | {row.method} | '+', '.join(diagnostics),fontsize=12)
    plt.tight_layout();folder=WORST/category if category in ['preserve_false_positive','remove_false_positive','high_ambiguity','staff','symbol','mixed','background'] else COMPONENT_CONTACTS
    path=folder/f'{rank:03d}_{page}_component_{int(row.component):04d}_{row.method}.png';plt.savefig(path,dpi=CROP_DPI,bbox_inches='tight');plt.close(fig);return path


## Page contact sheets


In [ ]:
page_files=[]
for ix,page in enumerate(sorted(cache_df.page.unique()),1):
    page_files.append(str(save_page_sheet(page)))
    print(ix,'/',cache_df.page.nunique(),page)
print('Page sheets:',len(page_files),PAGE_CONTACTS)


## Rank route-specific failures and ambiguity


In [ ]:
rankings=[]
def rank_and_save(df,category,column,ascending=True):
    subset=df.dropna(subset=[column]).sort_values(column,ascending=ascending).head(MAX_WORST_CROPS).copy()
    for rank,(_,row) in enumerate(subset.iterrows(),1):
        path=save_component_sheet(row,category,rank)
        rankings.append({'category':category,'rank':rank,'page':row.page,'component':int(row.component),'method':row.method,'ranking_metric':column,'ranking_value':float(row[column]),'output_file':str(path)})
    print(category,':',len(subset))

rank_and_save(component_df,'preserve_false_positive','preserved_clean_print_precision',True)
rank_and_save(component_df,'remove_false_positive','removed_clean_paper_precision',True)
rank_and_save(component_df,'high_ambiguity','reconstruction_dependency_fraction',False)


## Rank worst background, staff, symbol and mixed components


In [ ]:
def printed_ink(im): return cv2.adaptiveThreshold(im,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY_INV,41,15)>0
def edge_map(im): return cv2.Canny(im,70,170)>0
def tolerant_f1(ref,pred,region,tol=1):
    ref=ref&region;pred=pred&region
    if not ref.sum(): return np.nan
    k=np.ones((2*tol+1,2*tol+1),np.uint8)
    precision=np.sum(pred&(cv2.dilate(ref.astype(np.uint8),k)>0))/max(pred.sum(),1)
    recall=np.sum(ref&(cv2.dilate(pred.astype(np.uint8),k)>0))/ref.sum()
    return float(2*precision*recall/max(precision+recall,1e-12))
def clean_regions(clean,mask):
    ink=printed_ink(clean);long=cv2.morphologyEx(ink.astype(np.uint8),cv2.MORPH_OPEN,np.ones((1,35),np.uint8))>0
    staff=cv2.dilate(long.astype(np.uint8),np.ones((3,7),np.uint8))>0;near=cv2.dilate(staff.astype(np.uint8),np.ones((5,9),np.uint8))>0;m=mask>0
    return {'background':m&~ink,'staff':m&staff,'symbol':m&ink&~near,'mixed':m&ink&near}

def component_quality_rows():
    out=[]
    base=component_df[['page','method','component','x','y','w','h','area']].drop_duplicates()
    for page in sorted(base.page.unique()):
        clean,observed,a,_=load_page(page); mask=a['mask']; regions=clean_regions(clean,mask)
        n,lab,stats,_=cv2.connectedComponentsWithStats((mask>0).astype(np.uint8),8)
        for _,row in base[base.page==page].iterrows():
            cid=int(row.component)
            if cid<=0 or cid>=n: continue
            comp=lab==cid
            counts={k:int(np.sum(v&comp)) for k,v in regions.items()}; category=max(counts,key=counts.get)
            for method in ['geometry_t80','geometry_t110']:
                recon=a[method]; diff=np.abs(clean.astype(np.float32)-recon.astype(np.float32))
                out.append({**row.to_dict(),'method':method,'category':category,'component_mae':float(diff[comp].mean()),'component_ink_f1':tolerant_f1(printed_ink(clean),printed_ink(recon),comp),'component_edge_f1':tolerant_f1(edge_map(clean),edge_map(recon),comp)})
    return pd.DataFrame(out)

quality=component_quality_rows();quality.to_csv(TABLES/'final_slice_component_quality.csv',index=False)
for category,metric,ascending in [('background','component_mae',False),('staff','component_ink_f1',True),('symbol','component_edge_f1',True),('mixed','component_edge_f1',True)]:
    rank_and_save(quality[quality.category==category],category,metric,ascending)


## Additional component review sheets


In [ ]:
# One compact review sheet for the most structurally complex components at each threshold.
review_rows=(component_df.assign(structural_support=lambda d:d.horizontal_fraction+d.vertical_fraction+d.compact_fraction)
             .sort_values(['structural_support','reconstruction_dependency_fraction'],ascending=[False,False])
             .drop_duplicates(['page','method','component']).head(30))
review_files=[]
for rank,(_,row) in enumerate(review_rows.iterrows(),1):
    page=str(row.page);threshold=int(str(row.method).split('t')[-1]);clean,observed,a,_=load_page(page);routes=route_arrays(a,threshold);x0,y0,x1,y1=crop_bounds(row,clean.shape)
    overlay=route_overlay(observed,routes['preserve'],routes['remove'],routes['ambiguous'])
    fig,axes=plt.subplots(1,5,figsize=(20,4));panels=[clean[y0:y1,x0:x1],observed[y0:y1,x0:x1],a[row.method][y0:y1,x0:x1],overlay[y0:y1,x0:x1],(a['mask'][y0:y1,x0:x1]>0).astype(np.uint8)*255];titles=['clean','observed',row.method,'route overlay','mask']
    for ax,image,title in zip(axes,panels,titles):
        if image.ndim==2:show_gray(ax,image,title)
        else:ax.imshow(image);ax.set_title(title);ax.axis('off')
    fig.suptitle(f'{page} component {int(row.component)} | structural support={row.structural_support:.3f} | ambiguous={row.reconstruction_dependency_fraction:.3f}')
    plt.tight_layout();path=COMPONENT_CONTACTS/f'{rank:03d}_{page}_component_{int(row.component):04d}_{row.method}.png';plt.savefig(path,dpi=CROP_DPI,bbox_inches='tight');plt.close(fig);review_files.append(str(path))
print('Component review sheets:',len(review_files),COMPONENT_CONTACTS)


## Export final-slice manifest and display a sample


In [ ]:
ranking_df=pd.DataFrame(rankings);ranking_df.to_csv(TABLES/'final_slice_rankings.csv',index=False)
manifest={'created_utc':datetime.now(timezone.utc).isoformat(),'selected_v7_5_run':str(RUN),'reporting_only':True,'reconstruction_rerun':False,'pages':int(cache_df.page.nunique()),'page_contact_sheets':len(page_files),'component_review_sheets':len(review_files),'worst_crop_rows':len(ranking_df),'worst_crop_categories':sorted(ranking_df.category.unique().tolist()) if not ranking_df.empty else [],'outputs':{'page_comparisons':str(PAGE_CONTACTS),'component_review':str(COMPONENT_CONTACTS),'worst_crops':str(WORST),'rankings':str(TABLES/'final_slice_rankings.csv'),'component_quality':str(TABLES/'final_slice_component_quality.csv')}}
(RUN/'final_slice_manifest.json').write_text(json.dumps(manifest,indent=2))
print(json.dumps(manifest,indent=2))
for path in sorted(PAGE_CONTACTS.glob('*.png'))[:2]:
    im=cv2.imread(str(path));plt.figure(figsize=(15,20));plt.imshow(cv2.cvtColor(im,cv2.COLOR_BGR2RGB));plt.title(path.name);plt.axis('off');plt.show()
print('FINAL SLICE COMPLETE:',RUN)
